# Rice Variety Classification Using Logistic Regression

Clean portfolio notebook reconstructed from the original university Jupyter analysis. The analysis classifies **Cammeo** and **Osmancik** rice grains using seven physical measurements.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import arff
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, roc_auc_score


## Load the UCI Rice dataset
Place `Rice_Cammeo_Osmancik.arff` in the repository's `data/` folder before running the notebook.


In [ ]:
data, meta = arff.loadarff('../data/Rice_Cammeo_Osmancik.arff')
df = pd.DataFrame(data)
df['Class'] = df['Class'].str.decode('utf-8')
df.head()


## Exploratory data analysis
The dataset contains 3,810 observations. The original analysis examined class distribution, the Area feature, and correlations among the seven numerical grain measurements.


In [ ]:
df['Class'].value_counts()


In [ ]:
df['Class'].value_counts().plot(kind='bar')
plt.title('Rice Variety Distribution')
plt.xlabel('Rice Type')
plt.ylabel('Count')
plt.show()


In [ ]:
sns.boxplot(data=df, x='Class', y='Area')
plt.title('Area by Rice Variety')
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop(columns='Class').corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## Prepare the target and split the data
Cammeo is encoded as 0 and Osmancik as 1. The original assignment used a 70/30 train-test split and BT-ID **732084** as the random state.


In [ ]:
X = df.drop(columns='Class')
y = df['Class'].map({'Cammeo': 0, 'Osmancik': 1})

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=732084
)


## Train Logistic Regression


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4%}')


## Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


## ROC curve and ROC-AUC


In [ ]:
y_prob = model.predict_proba(x_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()
print(f'ROC-AUC: {auc_score:.4f}')


## 5-fold cross-validation


In [ ]:
scores = cross_val_score(
    LogisticRegression(max_iter=1000), X, y, cv=5, scoring='accuracy'
)
print('Fold accuracies:', scores)
print(f'Mean cross-validation accuracy: {scores.mean():.4%}')


## Results
The original completed analysis reported **93.53% test accuracy**, **0.9816 ROC-AUC**, and **93.02% mean 5-fold cross-validation accuracy**. These results indicate strong discrimination between Cammeo and Osmancik based on physical grain measurements.
